# Topology Metrics: Evaluating the Reasoning Trajectory in Title Optimization

This research notebook applies Topological Data Analysis (TDA) to understand how Large Language Models (LLMs) explore the semantic space of video titles. By using Persistent Homology and spatial partitioning, we evaluate the quality of initial suggestions and the efficiency of the iterative refinement process.

In [ ]:
# Setup and Dependencies
!pip install -q sentence-transformers ripser persim matplotlib seaborn

import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from ripser import ripser
from persim import plot_diagrams
from google.colab import drive
import matplotlib.animation as animation
from IPython.display import HTML
from scipy.spatial import ConvexHull
from matplotlib.path import Path

drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/numeric_inference_outputs/'
RESULTS_PATH = os.path.join(BASE_PATH, 'title_optimization_results.json')
EVAL_DATA_PATH = os.path.join(BASE_PATH, 'top_significant_channels_eval.json')
TRAIN_DATA_PATH = os.path.join(BASE_PATH, 'train_structured_latest.json')
EMBEDDING_CACHE_PATH = os.path.join(BASE_PATH, 'video_title_embeddings_latest.json')
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

with open(RESULTS_PATH, 'r') as f:
    results = json.load(f)

with open(EVAL_DATA_PATH, 'r') as f:
    eval_dataset = json.load(f)

with open(TRAIN_DATA_PATH, 'r') as f:
    train_data = json.load(f)

with open(EMBEDDING_CACHE_PATH, 'r') as f:
    embedding_cache = json.load(f)

# PCA Reconstruction for low-dimensional projection
all_train_embeddings = []
for channel in train_data:
    for video in channel['videos']:
        if video['title'] in embedding_cache:
            all_train_embeddings.append(embedding_cache[video['title']])

X_train = np.array(all_train_embeddings)
pca = PCA(n_components=15, random_state=42)
pca.fit(X_train)

df = pd.DataFrame(results)
print(f"Loaded {len(df)} optimization results.")

## Hypothesis 1: The topology of initial suggestions determines the potential for optimization

### Methodology
We use Persistent Homology to analyze the spatial distribution of the first batch of generated titles in the 2D latent space defined by the channel's most significant dimensions. We measure 'agglomeration' (H0 persistence) and 'spread' (H1 persistence) as the radius increases. We also correlate these topological features with final optimization success.

### Hypotheses
- **Null Hypothesis (H0)**: The initial spatial distribution of suggestions is random and does not correlate with final performance.
- **Alternative Hypothesis (H1)**: Suggestions that agglomerate around promising directions (high-weight dimensions) and maintain a balanced spread lead to better optimization outcomes.


In [ ]:
def get_top_2_dims(channel_name, eval_dataset):
    channel_data = next(c for c in eval_dataset if c['channel_name'].lower() == channel_name.lower())
    coeffs = np.abs(channel_data['model']['coefficients'])
    return np.argsort(coeffs)[-2:][::-1]

def analyze_initial_topology(video_row, channel_eval, pca, embedding_model):
    top_2 = get_top_2_dims(video_row['channel'], eval_dataset)
    
    # Get embeddings for original title and initial batch (iteration 0)
    orig_emb = embedding_model.encode([video_row['original_title']])
    it0_titles = [t['text'] for t in video_row['history'][0]['titles']]
    it0_scores = [t['score'] for t in video_row['history'][0]['titles']]
    it0_embs = embedding_model.encode(it0_titles)
    
    # Project to top 2 dims
    orig_proj = pca.transform(orig_emb)[0][top_2]
    it0_projs = pca.transform(it0_embs)[:, top_2]
    
    # Persistent Homology
    points = np.vstack([orig_proj, it0_projs])
    result = ripser(points)
    dgms = result['dgms']
    
    return points, dgms, it0_scores, top_2

def get_initial_features(video_row, channel_eval, pca, embedding_model):
    _, dgms, _, _ = analyze_initial_topology(video_row, channel_eval, pca, embedding_model)
    h0 = dgms[0]
    h1 = dgms[1]
    # Agglomeration: mean death time of components (lower means points merge faster)
    agglomeration = np.mean(h0[:-1, 1]) if len(h0) > 1 else 0
    # Spread: total persistence of loops/voids
    spread = np.sum(h1[:, 1] - h1[:, 0]) if len(h1) > 0 else 0
    return {
        'agglomeration': agglomeration,
        'spread': spread,
        'improvement': video_row['improvement']
    }

# Sample Video Analysis
sample_row = df.iloc[0]
channel_eval = next(c for c in eval_dataset if c['channel_name'].lower() == sample_row['channel'].lower())
points, dgms, scores, dims = analyze_initial_topology(sample_row, channel_eval, pca, embedding_model)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(points[1:, 0], points[1:, 1], c=scores, cmap='RdYlGn', label='Initial Batch')
plt.scatter(points[0, 0], points[0, 1], c='blue', s=100, marker='X', label='Original')
plt.title(f"Initial Suggestions Space (Dims {dims})")
plt.colorbar(label='Predicted Score')
plt.legend()

plt.subplot(1, 2, 2)
plot_diagrams(dgms, show=True)
plt.show()


### Animation: Growing Radius Convergence
We visualize how the topology evolves as we increase the 'radius' around each suggestion. Red/Green represents performance, Blue is the original title.

In [ ]:
def create_radius_animation(points, scores, filename='topology_evolution.mp4'):
    fig, ax = plt.subplots(figsize=(8, 8))
    norm = plt.Normalize(min(scores), max(scores))
    colors = [plt.cm.RdYlGn(norm(s)) for s in scores]
    colors = ['blue'] + colors
    
    def update(r):
        ax.clear()
        padding = 0.5
        ax.set_xlim(points[:, 0].min() - padding, points[:, 0].max() + padding)
        ax.set_ylim(points[:, 1].min() - padding, points[:, 1].max() + padding)
        for i, (p, c) in enumerate(zip(points, colors)):
            circle = plt.Circle((p[0], p[1]), r, color=c, alpha=0.2)
            ax.add_patch(circle)
            ax.scatter(p[0], p[1], color=c, s=30)
        ax.set_title(f"TDA Radius Expansion: {r:.2f}")

    radii = np.linspace(0, 0.4, 40)
    ani = animation.FuncAnimation(fig, update, frames=radii, interval=150)
    save_path = os.path.join(BASE_PATH, filename)
    ani.save(save_path, writer='ffmpeg')
    plt.close()
    return save_path

video_path = create_radius_animation(points, scores)
print(f"Animation saved to {video_path}")

### Correlation Analysis for Initial Topology

In [ ]:
initial_metrics = []
for _, row in df.iterrows():
    c_eval = next(c for c in eval_dataset if c['channel_name'].lower() == row['channel'].lower())
    initial_metrics.append(get_initial_features(row, c_eval, pca, embedding_model))

init_df = pd.DataFrame(initial_metrics)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.regplot(data=init_df, x='agglomeration', y='improvement')
r_agg, _ = stats.pearsonr(init_df['agglomeration'], init_df['improvement'])
plt.title(f'Agglomeration vs Improvement (r={r_agg:.2f})')

plt.subplot(1, 2, 2)
sns.regplot(data=init_df, x='spread', y='improvement')
r_spr, _ = stats.pearsonr(init_df['spread'], init_df['improvement'])
plt.title(f'Initial Spread vs Improvement (r={r_spr:.2f})')
plt.show()

### Interpretation
The results indicate that...

## Hypothesis 2: Iterative suggestions exploit previously defined semantic voids

### Methodology
We define a 'semantic void' as the convex hull of the top performing suggestions from previous iterations. We analyze the trajectory of new suggestions to determine if they fall within these 'explored' partitions (exploitation), or if they venture into 'unexplored' territory (exploration).

### Hypotheses
- **Null Hypothesis (H0)**: New iterations do not show a preference for previously successful semantic partitions.
- **Alternative Hypothesis (H1)**: LLM reasoning converges toward high-performing 'voids' identified in earlier rounds.


In [ ]:
def analyze_iterative_quality(video_row, pca, embedding_model):
    """
    Methodology:
    1. For each iteration, define the 'Explored Void' as the Convex Hull of top 3 
       performers from ALL previous iterations.
    2. Define 'New Territory' as suggestions falling outside this Hull.
    """
    top_2 = get_top_2_dims(video_row['channel'], eval_dataset)
    void_stats = []
    historical_top_points = []
    
    for i, iteration in enumerate(video_row['history']):
        titles = [t['text'] for t in iteration['titles']]
        scores = [t['score'] for t in iteration['titles']]
        projs = pca.transform(embedding_model.encode(titles))[:, top_2]
        
        inside_count = 0
        if i > 0 and len(historical_top_points) >= 3:
            try:
                hull = ConvexHull(np.array(historical_top_points))
                hull_path = Path(np.array(historical_top_points)[hull.vertices])
                inside_count = sum(hull_path.contains_points(projs))
            except:
                inside_count = 0
            
        void_stats.append({
            'iteration': i,
            'inside_void': inside_count,
            'new_territory': len(projs) - inside_count,
            'avg_score': np.mean(scores)
        })
        
        top_indices = np.argsort(scores)[-3:]
        historical_top_points.extend(projs[top_indices].tolist())
        
    return pd.DataFrame(void_stats)

iter_quality_results = []
for _, row in df.iterrows():
    v_stats = analyze_iterative_quality(row, pca, embedding_model)
    # Metric: Percentage of titles exploiting the known 'good' space
    num_titles = len(row['history'][0]['titles'])
    v_stats['pct_inside'] = v_stats['inside_void'] / num_titles
    
    iter_quality_results.append({
        'video_id': row['video_id'],
        'avg_exploitation': v_stats['pct_inside'].mean(),
        'exploitation_trend': stats.linregress(v_stats['iteration'], v_stats['pct_inside']).slope,
        'improvement': row['improvement']
    })

iter_qual_df = pd.DataFrame(iter_quality_results)
display(iter_qual_df.describe())

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.regplot(data=iter_qual_df, x='avg_exploitation', y='improvement')
plt.title('Avg Exploitation vs Improvement')
plt.subplot(1, 2, 2)
sns.regplot(data=iter_qual_df, x='exploitation_trend', y='improvement')
plt.title('Exploitation Trend vs Improvement')
plt.show()

# TODO: Implement N-dimensional point-in-poly check to move beyond 2D projections.
# TODO: Explore Alpha Shapes for a tighter definition of 'known good' space.


### Interpretation
The analysis shows that videos with a positive exploitation trend...

## Hypothesis 3: Topological Coverage is a superior predictor of optimization potential

### Methodology
We define 'Coverage' using multi-dimensional topological metrics:
1. **Connectedness**: The number of independent semantic clusters explored.
2. **Persistent Loops (H1)**: The structural complexity and 'holes' in the reasoning trajectory.
3. **Performance Density**: The ratio of high-performing connected components.
4. **Evolution Drift**: The total distance moved from the original semantic anchor.

### Hypotheses
- **Null Hypothesis (H0)**: Traditional variance-based coverage is as effective as topological coverage.
- **Alternative Hypothesis (H1)**: Topological metrics provide a more nuanced and predictive measure of exploration.


In [ ]:
def calculate_coverage_metrics(video_row, pca, embedding_model):
    top_2 = get_top_2_dims(video_row['channel'], eval_dataset)
    all_titles = []
    for it in video_row['history']:
        all_titles.extend([t['text'] for t in it['titles']])
    
    projs = pca.transform(embedding_model.encode(all_titles))[:, top_2]
    result = ripser(projs)
    h0 = result['dgms'][0]
    h1 = result['dgms'][1]
    
    # Metrics
    r_thresh = 0.1
    connectedness = np.sum(h0[:, 1] > r_thresh) if len(h0) > 0 else 0
    persistence_sum_h1 = np.sum(h1[:, 1] - h1[:, 0]) if len(h1) > 0 else 0
    
    all_scores = []
    for it in video_row['history']:
        all_scores.extend([t['score'] for t in it['titles']])
    perf_density = np.mean(np.array(all_scores) >= np.percentile(all_scores, 75))
    
    orig_proj = pca.transform(embedding_model.encode([video_row['original_title']]))[0][top_2]
    drift = np.mean(np.linalg.norm(projs - orig_proj, axis=1))

    return {
        'video_id': video_row['video_id'],
        'connectedness': connectedness,
        'h1_total_persistence': persistence_sum_h1,
        'performance_density': perf_density,
        'evolution_drift': drift,
        'improvement': video_row['improvement']
    }

coverage_results = []
for _, row in df.iterrows():
    coverage_results.append(calculate_coverage_metrics(row, pca, embedding_model))

cov_df = pd.DataFrame(coverage_results)

plt.figure(figsize=(15, 10))
metrics_to_plot = ['connectedness', 'h1_total_persistence', 'performance_density', 'evolution_drift']
for i, metric in enumerate(metrics_to_plot):
    plt.subplot(2, 2, i+1)
    sns.regplot(data=cov_df, x=metric, y='improvement')
    r, _ = stats.pearsonr(cov_df[metric], cov_df['improvement'])
    plt.title(f'{metric.capitalize()} vs Improvement (r={r:.2f})')
plt.tight_layout()
plt.show()

print("### Summary Coverage Table")
display(cov_df.drop(columns=['video_id']).corr()[['improvement']])


### Evolution Table: Iterative Metrics
We track how the average distance to the original title and average score evolve.

In [ ]:
evolution_data = []
for _, row in df.iterrows():
    top_2 = get_top_2_dims(row['channel'], eval_dataset)
    orig_proj = pca.transform(embedding_model.encode([row['original_title']]))[0][top_2]
    for i, it in enumerate(row['history']):
        it_projs = pca.transform(embedding_model.encode([t['text'] for t in it['titles']]))[:, top_2]
        avg_dist = np.mean(np.linalg.norm(it_projs - orig_proj, axis=1))
        avg_score = np.mean([t['score'] for t in it['titles']])
        evolution_data.append({'video_id': row['video_id'], 'iteration': i, 'avg_dist': avg_dist, 'avg_score': avg_score})

evol_df = pd.DataFrame(evolution_data)
plt.figure(figsize=(10, 6))
sns.lineplot(data=evol_df, x='iteration', y='avg_dist', hue='video_id', legend=False, alpha=0.3)
plt.title('Evolution of Semantic Distance from Original Title')
plt.show()


## Conclusions and Future Work

Topological metrics provide a powerful lens to view the optimization process...

In [ ]:
# Save final results to Drive
cov_df.to_json(os.path.join(BASE_PATH, 'topology_metrics_results.json'))
